In [1]:
import os
import struct
import pandas as pd
import matplotlib.pyplot as plt

# --- Step 1: The NEW, MORE ROBUST Decoder Function ---
def read_ai_file_debug(path_to_ai_file):
    """
    A more robust version of the AI reader that provides detailed debugging
    information about the file's header and structure.
    """
    print(f"--- Running Diagnostic Read on: {path_to_ai_file} ---")
    
    try:
        with open(path_to_ai_file, "rb") as f:
            # --- File Size Check ---
            total_file_size = os.path.getsize(path_to_ai_file)
            print(f"Total file size is: {total_file_size} bytes.")
            if total_file_size < 16:
                print("ERROR: File is too small to be a valid AI line file.")
                return None

            # --- Header Inspection ---
            # Let's read the first 16 bytes (the header) and print them
            header_bytes = f.read(16)
            # 'hex()' converts the bytes to a more readable hexadecimal string
            print(f"Header (first 16 bytes): {header_bytes.hex()}")

            # The point count is supposed to be at byte offset 12
            # We use struct.unpack_from to read without moving the file pointer
            count = struct.unpack_from("i", header_bytes, 12)[0]
            print(f"DEBUG: Value read for point count is: {count}")

            if count <= 0:
                print("ERROR: The file header indicates 0 or fewer points.")
                print("This is the likely cause of the issue. The file may not be a 'fast_lane.ai' file.")
                return None

            # --- Data Consistency Check ---
            expected_size = 16 + (count * 24) # 16-byte header + 24 bytes per point
            print(f"Based on count, expected file size is: {expected_size} bytes.")
            if total_file_size != expected_size:
                print("WARNING: Actual file size does not match expected size. The format may be different.")

            # --- Data Reading ---
            points = []
            f.seek(16) # Go to the start of the data points
            for i in range(count):
                # Read the 24 bytes for one point
                point_data = f.read(24)
                if len(point_data) < 24:
                    print(f"WARNING: Reached end of file prematurely at point {i+1}.")
                    break
                
                x, y, z, _, _, _ = struct.unpack("ffffff", point_data)
                points.append([x, z, y])

            return points

    except FileNotFoundError:
        print(f"ERROR: File not found at '{path_to_ai_file}'")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# --- Step 2: Main Execution Block ---
if __name__ == "__main__":
    track_folder_name = "Tracks"
    track_file_name = "silverstone_gp.ai"
    
    current_directory = os.getcwd()
    file_path = os.path.join(current_directory, track_folder_name, track_file_name)

    # Call the new diagnostic function
    track_points = read_ai_file_debug(file_path)

    if track_points:
        track_df = pd.DataFrame(track_points, columns=['x', 'y', 'z'])
        print("\nSuccessfully created pandas DataFrame:")
        print(track_df.head())

        print("\nGenerating 3D plot...")
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(projection='3d')
        plot_title = os.path.splitext(track_file_name)[0].replace("_", " ").title()
        ax.plot(track_df['x'], track_df['y'], track_df['z'], label='Fast Lane')
        ax.set_title(f'3D Track Map: {plot_title}')
        ax.set_xlabel('X Coordinate')
        ax.set_ylabel('Y Coordinate (Depth)')
        ax.set_zlabel('Z Coordinate (Elevation)')
        ax.set_aspect('auto')
        ax.legend()
        ax.grid(True)
        plt.show()

--- Running Diagnostic Read on: h:\My Drive\Cranfield\11. Individual Thesis\Code\Tracks\silverstone_gp.ai ---
Total file size is: 2159460 bytes.
Header (first 16 bytes): 070000001e0f00000000000000000000
DEBUG: Value read for point count is: 0
ERROR: The file header indicates 0 or fewer points.
This is the likely cause of the issue. The file may not be a 'fast_lane.ai' file.
